In [1]:
import requests
import xml.etree.ElementTree as ET

/Users/ekta/Desktop/Sabudh Internship/ResearchMate/Code/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
query = "machine learning healthcare"

In [3]:
search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": query,
    "retmode": "json",
    "retmax": 5
}

In [4]:
response = requests.get(search_url, params=params)

In [5]:
print(response.status_code)

200


In [6]:
print(response.url)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=machine+learning+healthcare&retmode=json&retmax=5


In [7]:
data = response.json()

In [8]:
data

{'header': {'type': 'esearch', 'version': '0.3'},
 'esearchresult': {'count': '23115',
  'retmax': '5',
  'retstart': '0',
  'idlist': ['42864330', '42863254', '42862568', '42862311', '42861755'],
  'translationset': [{'from': 'machine learning',
    'to': '"machine learning"[MeSH Terms] OR ("machine"[All Fields] AND "learning"[All Fields]) OR "machine learning"[All Fields]'},
   {'from': 'healthcare',
    'to': '"delivery of health care"[MeSH Terms] OR ("delivery"[All Fields] AND "health"[All Fields] AND "care"[All Fields]) OR "delivery of health care"[All Fields] OR "healthcare"[All Fields] OR "healthcare\'s"[All Fields] OR "healthcares"[All Fields]'}],
  'querytranslation': '("machine learning"[MeSH Terms] OR ("machine"[All Fields] AND "learning"[All Fields]) OR "machine learning"[All Fields]) AND ("delivery of health care"[MeSH Terms] OR ("delivery"[All Fields] AND "health"[All Fields] AND "care"[All Fields]) OR "delivery of health care"[All Fields] OR "healthcare"[All Fields] OR "

In [9]:
data["esearchresult"]["count"]
data["esearchresult"]["idlist"]

['42864330', '42863254', '42862568', '42862311', '42861755']

In [10]:
print("Total matches:", data["esearchresult"]["count"])
print("Returned PMIDs:", data["esearchresult"]["idlist"])

Total matches: 23115
Returned PMIDs: ['42864330', '42863254', '42862568', '42862311', '42861755']


In [11]:
pmids = data["esearchresult"]["idlist"]
print(pmids)

['42864330', '42863254', '42862568', '42862311', '42861755']


In [12]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

fetch_params = {
    "db": "pubmed",
    "id": ",".join(pmids),
    "retmode": "xml"
}

In [13]:
fetch_response = requests.get(fetch_url, params=fetch_params)

print(fetch_response.status_code)

200


In [14]:
print(fetch_response.url)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id=42864330%2C42863254%2C42862568%2C42862311%2C42861755&retmode=xml


In [15]:
print(fetch_response.text[:3000])

<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" IndexingMethod="Automated"><PMID Version="1">42864330</PMID><DateCompleted><Year>2026</Year><Month>10</Month><Day>12</Day></DateCompleted><DateRevised><Year>2026</Year><Month>10</Month><Day>12</Day></DateRevised><Article PubModel="Electronic"><Journal><ISSN IssnType="Electronic">2041-1723</ISSN><JournalIssue CitedMedium="Internet"><Volume>17</Volume><Issue>1</Issue><PubDate><Year>2026</Year><Month>Sep</Month><Day>10</Day></PubDate></JournalIssue><Title>Nature communications</Title><ISOAbbreviation>Nat Commun</ISOAbbreviation></Journal><ArticleTitle>Assessing the use of automatic speech recognition and large language models for individuals with language impairments.</ArticleTitle><ELocationID EIdType="pii" ValidYN="Y">10716</ELocationID><ELoca

In [16]:
root = ET.fromstring(fetch_response.content)
articles = root.findall(".//PubmedArticle")
print(len(articles))

5


In [17]:
article = articles[0]

In [18]:
title_node = article.find(".//ArticleTitle")
title = "".join(title_node.itertext()) if title_node is not None else None
print(title)

Assessing the use of automatic speech recognition and large language models for individuals with language impairments.


In [19]:
abstract_nodes = article.findall(".//Abstract/AbstractText")

abstract_parts = []

for node in abstract_nodes:
    text = "".join(node.itertext())
    label = node.attrib.get("Label")

    if label:
        abstract_parts.append(f"{label}: {text}")
    else:
        abstract_parts.append(text)

abstract = " ".join(abstract_parts) if abstract_parts else None

print(abstract)

Large language models (LLMs) have attracted much attention for healthcare applications, demonstrating strong potential in automating conversational interactions. However, cloud-hosted systems raise privacy concerns and text-based interfaces can be difficult for children, older adults, and individuals with language impairments. Edge-deployed, voice-enabled LLM systems may address these barriers by keeping data local and using automatic speech recognition (ASR) for speech input. However, ASR transcripts often contain disfluencies, fillers, grammatical errors, and recognition noise, which may reduce downstream LLM performance for speakers with atypical language patterns. Here, we systematically evaluate edge-oriented ASR-LLM pipelines for individuals with language impairments using comparison studies and ablation experiments across aphasia, child language impairment, and dementia datasets. We identify transcript errors, repetition, noise, and input length as factors affecting system perfo

In [20]:
year_node = article.find(".//JournalIssue/PubDate/Year")

year = year_node.text if year_node is not None else None

print(year)

2026


In [21]:
pmid = article.findtext(".//PMID")

print(pmid)

42864330


In [22]:
doi = None

for node in article.findall(".//ELocationID"):
    if node.attrib.get("EIdType") == "doi":
        doi = node.text
        break

print(doi)

10.1038/s41467-026-76677-z


In [23]:
print("PMID:", pmid)
print("Title:", title)
print("Abstract:", abstract)
print("Year:", year)
print("DOI:", doi)

PMID: 42864330
Title: Assessing the use of automatic speech recognition and large language models for individuals with language impairments.
Abstract: Large language models (LLMs) have attracted much attention for healthcare applications, demonstrating strong potential in automating conversational interactions. However, cloud-hosted systems raise privacy concerns and text-based interfaces can be difficult for children, older adults, and individuals with language impairments. Edge-deployed, voice-enabled LLM systems may address these barriers by keeping data local and using automatic speech recognition (ASR) for speech input. However, ASR transcripts often contain disfluencies, fillers, grammatical errors, and recognition noise, which may reduce downstream LLM performance for speakers with atypical language patterns. Here, we systematically evaluate edge-oriented ASR-LLM pipelines for individuals with language impairments using comparison studies and ablation experiments across aphasia,

In [24]:
def parse_pubmed_article(article, query):
    title_node = article.find(".//ArticleTitle")
    title = "".join(title_node.itertext()) if title_node is not None else None

    abstract_nodes = article.findall(".//Abstract/AbstractText")
    abstract_parts = []

    for node in abstract_nodes:
        text = "".join(node.itertext()).strip()
        label = node.attrib.get("Label")

        if label:
            abstract_parts.append(f"{label}: {text}")
        else:
            abstract_parts.append(text)

    abstract = " ".join(abstract_parts) if abstract_parts else None

    year_node = article.find(".//JournalIssue/PubDate/Year")
    year = year_node.text if year_node is not None else None

    pmid = article.findtext(".//PMID")

    doi = None
    for node in article.findall(".//ELocationID"):
        if node.attrib.get("EIdType") == "doi":
            doi = node.text
            break
    authors = []

    for author in article.findall(".//Author"):
        first = author.findtext("ForeName")
        last = author.findtext("LastName")

        if first or last:
            name = " ".join(
                part for part in [first, last]
                if part
            )
            authors.append(name)

    keywords = [
        "".join(node.itertext()).strip()
        for node in article.findall(".//Keyword")]

    return {
    "query": query,
    "source": "PubMed",
    "source_id": pmid,
    "title": title,
    "authors": authors,
    "abstract": abstract,
    "keywords": keywords,
    "year": year,
    "doi": doi}

In [25]:
pubmed_papers = []

for article in articles:
    paper = parse_pubmed_article(article, query)
    pubmed_papers.append(paper)

In [26]:
for paper in pubmed_papers:
    print(paper)
    print("-" * 80)

{'query': 'machine learning healthcare', 'source': 'PubMed', 'source_id': '42864330', 'title': 'Assessing the use of automatic speech recognition and large language models for individuals with language impairments.', 'authors': ['Gelei Xu', 'Haoxinran Yu', 'Lixuan Wei', 'Yuxuan Liu', 'Dancheng Liu', 'Chenhui Xu', 'Jiajie Li', 'Ahmed Abbasi', 'Jinjun Xiong', 'Xiufan Yu', 'Zhi Zheng', 'Yiyu Shi', 'Ruiyang Qin'], 'abstract': 'Large language models (LLMs) have attracted much attention for healthcare applications, demonstrating strong potential in automating conversational interactions. However, cloud-hosted systems raise privacy concerns and text-based interfaces can be difficult for children, older adults, and individuals with language impairments. Edge-deployed, voice-enabled LLM systems may address these barriers by keeping data local and using automatic speech recognition (ASR) for speech input. However, ASR transcripts often contain disfluencies, fillers, grammatical errors, and recog

## arXIV API


In [27]:
arxiv_url = "http://export.arxiv.org/api/query"

In [28]:
query = "machine learning healthcare"

In [29]:
arxiv_params = {
    "search_query": f"all:{query}",
    "start": 0,
    "max_results": 5
}

In [30]:
arxiv_response = requests.get(arxiv_url, params=arxiv_params)

print(arxiv_response.status_code)
print(arxiv_response.url)

200
https://export.arxiv.org/api/query?search_query=all%3Amachine+learning+healthcare&start=0&max_results=5


In [31]:
print(arxiv_response.text[:3000])

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/1bRQFiWTzjUJgMY7nsAHlnE42lo</id>
  <title>arXiv Query: search_query=all:machine OR all:learning OR all:healthcare&amp;id_list=&amp;start=0&amp;max_results=5</title>
  <updated>2026-10-13T16:06:47Z</updated>
  <link href="https://arxiv.org/api/query?search_query=all:machine+OR+(all:learning+OR+all:healthcare)&amp;start=0&amp;max_results=5&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>5</opensearch:itemsPerPage>
  <opensearch:totalResults>549361</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2303.15563v1</id>
    <title>Privacy-preserving machine learning for healthcare: open challenges and future perspectives</title>
    <updated>2023-03-27T19:20:51Z</updated>
    <link href="https://arxi

In [32]:
root = ET.fromstring(arxiv_response.content)

In [33]:
ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom"
}

In [34]:
entries = root.findall("atom:entry", ns)

print(len(entries))

5


In [35]:
entry = entries[0]

In [36]:
title = entry.findtext("atom:title", default=None, namespaces=ns)

print(title)

Privacy-preserving machine learning for healthcare: open challenges and future perspectives


In [37]:
abstract = entry.findtext("atom:summary", default=None, namespaces=ns)

print(abstract)

Machine Learning (ML) has recently shown tremendous success in modeling various healthcare prediction tasks, ranging from disease diagnosis and prognosis to patient treatment. Due to the sensitive nature of medical data, privacy must be considered along the entire ML pipeline, from model training to inference. In this paper, we conduct a review of recent literature concerning Privacy-Preserving Machine Learning (PPML) for healthcare. We primarily focus on privacy-preserving training and inference-as-a-service, and perform a comprehensive review of existing trends, identify challenges, and discuss opportunities for future research directions. The aim of this review is to guide the development of private and efficient ML models in healthcare, with the prospects of translating research efforts into real-world settings.


In [38]:
authors = [
    author.findtext("atom:name", default=None, namespaces=ns)
    for author in entry.findall("atom:author", ns)
]

print(authors)

['Alejandro Guerra-Manzanares', 'L. Julian Lechuga Lopez', 'Michail Maniatakos', 'Farah E. Shamout']


In [39]:
published = entry.findtext("atom:published", default=None, namespaces=ns)

print(published)

2023-03-27T19:20:51Z


In [40]:
year = published[:4] if published else None

print(year)

2023


In [41]:
doi = entry.findtext("arxiv:doi", default=None, namespaces=ns)

print(doi)

10.1007/978-3-031-39539-0_3


In [42]:
categories = [
    node.attrib.get("term")
    for node in entry.findall("atom:category", ns)
]

print(categories)

['cs.LG', 'cs.CR']


In [43]:
source_id = entry.findtext("atom:id", default=None, namespaces=ns)

print(source_id)

http://arxiv.org/abs/2303.15563v1


In [44]:
paper = {
    "query": query,
    "source": "arXiv",
    "source_id": source_id,
    "title": title,
    "authors": authors,
    "abstract": abstract,
    "keywords": categories,
    "year": year,
    "doi": doi
}

paper

{'query': 'machine learning healthcare',
 'source': 'arXiv',
 'source_id': 'http://arxiv.org/abs/2303.15563v1',
 'title': 'Privacy-preserving machine learning for healthcare: open challenges and future perspectives',
 'authors': ['Alejandro Guerra-Manzanares',
  'L. Julian Lechuga Lopez',
  'Michail Maniatakos',
  'Farah E. Shamout'],
 'abstract': 'Machine Learning (ML) has recently shown tremendous success in modeling various healthcare prediction tasks, ranging from disease diagnosis and prognosis to patient treatment. Due to the sensitive nature of medical data, privacy must be considered along the entire ML pipeline, from model training to inference. In this paper, we conduct a review of recent literature concerning Privacy-Preserving Machine Learning (PPML) for healthcare. We primarily focus on privacy-preserving training and inference-as-a-service, and perform a comprehensive review of existing trends, identify challenges, and discuss opportunities for future research directions.